## Programa para Adquirir Dados

In [1]:
import time
import clr
import numpy as np
import time 
from pylablib.devices import Thorlabs
from structured_optics import *
from structured_optics.hologram import slm_hologram
from structured_optics.slmdisplay import My_SLM
from tqdm import tqdm
from camera import *
from utils import *
import h5py
from pathlib import Path

# Add References to .NET libraries

clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.DeviceManagerCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.GenericMotorCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\ThorLabs.MotionControl.KCube.StepperMotorCLI.dll")
from Thorlabs.MotionControl.DeviceManagerCLI import *
from Thorlabs.MotionControl.GenericMotorCLI import *
from Thorlabs.MotionControl.KCube.StepperMotorCLI import *
from System import Decimal  # necessary for real world units


## Funções

In [2]:
def calculate_fix_std_LG(C, l, p, z):
    return np.sqrt(2*C**2/(2*p + np.abs(l) +1) + 2*np.sqrt(np.abs(C**4/(2*p + np.abs(l) +1)**2 - z**2/beam.k()**2)))/mu

def calculate_fix_std_HG(C, n, z):
    return np.sqrt(2*C**2/(2*n +1) + 2*np.sqrt(np.abs(C**4/(2*n +1)**2 - z**2/beam.k()**2)))/mu

## Parâmetros

In [ ]:
FOLDER = 'Amostras_10_2026/'


#parametros laser e slm
pixel_size = 8e-6  #tamanho do pixel do slm 
lamb = 633e-9   #comprimento de onda
waist = 0.3e-3  #cintura do feixe que sai do slm
x_grating = 3
y_grating = 4

#parametros lentes e propagação
mu = 0.6930012799317185/0.5 #parâmetro de magnificação devido ao sistema 4F após o SLM
z = 0 #Distância entre o waist pós 4F e a amostra, utilizado para calcular fix std
C = 0.65e-3#0.5e-3  #std a ser fixado


#parametros motor de passo
resolution = 50  #Resolução dos motores de passo
step_size = 200e-9 #tamanho do passo dos motores de passo
NUM_REP = 5  #numero de repetições da medida
z_start = 0.004

#parametros camera
EXPO = 1/9091   #exposição camera em segundos
BIT = 8     #bitdepth da camera
pixel_cam = 3.75e-6 #tamanho do pixel da camera
width_cam = 1280 #largura da camera
height_cam = 960 #altura da camera
NUM_BACK = 5 #numero de imagens para calcular o background


#parametros amostra
AMOSTRA = "Ferro"   #tipo de amostra
ANGULO_AMOSTRA = 90 #angulo das ranhuras. 90 representa vertical, 45 diagonal e 0 horizontal.



In [ ]:
"""folder = Path(FOLDER)
folder.mkdir(exist_ok=True)""" #cria a pasta onde salvamos os dados, só precisa ser rodado uma vez, por isso está comentado.

## Primeiro alinhamento e check de equipamentos

In [ ]:
#Inicializa e set home dos motores de passo

stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
print(f" stage z  -- {stage_z.get_position(scale= 'ZST')}")
stages = [stage_z]

for _ in stages:
    print(_.is_homed())
    if _.is_homed() == False:
        _.home()
for _ in stages:
    _.wait_for_home()


In [ ]:
#inicializa camera, slm e lista de modos a serem medidos (até ordem 5)

slm = My_SLM(1)
width = slm.width
height = slm.height

cam = init_camera(bit_depth=BIT, exposure_s = EXPO)
take_picture(cam)               # discard possible stale frame
frame = take_picture(cam)


hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], 
            [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], 
            [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
lg_modes = [[0,0], [0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], 
            [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], 
            [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]

In [ ]:
#inicializa objeto beam que contém os parâmetros do feixe gerado

width = slm.width
height = slm.height

beam = Beam(nix = pixel_size*width/2, Dx = width,
            niy = pixel_size*height/2, Dy = height,
            waist = waist, lamb = lamb)

In [ ]:
#Define qual o campo do objeto beam, gera um holograma a partir desse campo e atualiza a máscara do slm com esse holograma

beam.lg(2,2)
#beam.triangle() #usado para encontrar o plano do slm, visto que um triângulo difrata muito rápido
holo = slm_hologram(beam, x_grating, y_grating)
slm.update_image(holo)

## Dados LG waist normal

In [ ]:
beam.waist = waist

with h5py.File(f"{FOLDER}data_LG_amostra{AMOSTRA}_orientacao{ANGULO_AMOSTRA}.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'amostra': str(AMOSTRA),
        'angulo_amostra_deg': ANGULO_AMOSTRA,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })

    slm.update_image(np.zeros((height, width), dtype='uint8'))
    time.sleep(0.2)
    background = file.create_dataset("background", (NUM_BACK, height_cam, width_cam), dtype='uint8')
    for i in range(NUM_BACK):
        background[i] = take_picture(cam)
        time.sleep(0.1)

    for mode in tqdm(lg_modes, desc="Processing LG modes"):

        l = mode[1]
        p = mode[0]
        mode_group = file.create_group(f"LG_p{p}l{l}")
        images_ds = mode_group.create_dataset("images", (NUM_REP, resolution,height_cam, width_cam), dtype='uint8')

        beam.lg(l,p)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)
        for rep in range(NUM_REP):
            stage_z.move_to(z_start-0.001) # in meter
            stage_z.wait_move()
            stage_z.move_to(z_start) # in meter
            stage_z.wait_move()
            for i in range(resolution):
                stage_z.move_to(z_start + i*step_size)
                stage_z.wait_move()
                time.sleep(0.1)
                img = take_picture(cam)
                images_ds[rep, i] = img
            file.flush()


## Dados HG waist normal

In [ ]:
beam.waist = waist

with h5py.File(f"{FOLDER}data_HG_amostra{AMOSTRA}_orientacao{ANGULO_AMOSTRA}.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'amostra': str(AMOSTRA),
        'angulo_amostra_deg': ANGULO_AMOSTRA,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })
    
    slm.update_image(np.zeros((height, width), dtype='uint8'))
    time.sleep(0.2)
    background = file.create_dataset("background", (NUM_BACK, height_cam, width_cam), dtype='uint8')
    for i in range(NUM_BACK):
        background[i] = take_picture(cam)
        time.sleep(0.1)


    for n in tqdm(range(6), desc="Processing HG modes"):
        mode_group = file.create_group(f"HG_n{n}m{0}")
        images_ds = mode_group.create_dataset("images", (NUM_REP, resolution,height_cam, width_cam), dtype='uint8')

        beam.hg(n,0)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)
        for rep in range(NUM_REP):
            stage_z.move_to(z_start-0.001) # in meter
            stage_z.wait_move()
            stage_z.move_to(z_start) # in meter
            stage_z.wait_move()
            for i in range(resolution):
                stage_z.move_to(z_start + i*step_size)
                stage_z.wait_move()
                time.sleep(0.1)
                img = take_picture(cam)
                images_ds[rep, i] = img
            file.flush()

## Dados LG std fixo

In [ ]:

with h5py.File(f"{FOLDER}data_LG_fix_amostra{AMOSTRA}_orientacao{ANGULO_AMOSTRA}.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'amostra': str(AMOSTRA),
        'angulo_amostra_deg': ANGULO_AMOSTRA,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
        'C': C
    })

    slm.update_image(np.zeros((height, width), dtype='uint8'))
    time.sleep(0.2)
    background = file.create_dataset("background", (NUM_BACK, height_cam, width_cam), dtype='uint8')
    for i in range(NUM_BACK):
        background[i] = take_picture(cam)
        time.sleep(0.1)


    for mode in tqdm(lg_modes, desc="Processing LG fix std modes"):

        l = mode[1]
        p = mode[0]
        mode_group = file.create_group(f"LG_p{p}l{l}")
        images_ds = mode_group.create_dataset("images", (NUM_REP, resolution,height_cam, width_cam), dtype='uint8')

        beam.waist = calculate_fix_std_LG(C, l, p, z)
        beam.lg(l,p)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)
        for rep in range(NUM_REP):
            stage_z.move_to(z_start-0.001) # in meter
            stage_z.wait_move()
            stage_z.move_to(z_start) # in meter
            stage_z.wait_move()
            for i in range(resolution):
                stage_z.move_to(z_start + i*step_size)
                stage_z.wait_move()
                time.sleep(0.1)
                img = take_picture(cam)
                images_ds[rep, i] = img
            file.flush()

## Dados HG std fixo

In [ ]:

with h5py.File(f"{FOLDER}data_HG_fix_amostra{AMOSTRA}_orientacao{ANGULO_AMOSTRA}.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'amostra': str(AMOSTRA),
        'angulo_amostra_deg': ANGULO_AMOSTRA,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
        'C': C
    })

    slm.update_image(np.zeros((height, width), dtype='uint8'))
    time.sleep(0.2)
    background = file.create_dataset("background", (NUM_BACK, height_cam, width_cam), dtype='uint8')
    for i in range(NUM_BACK):
        background[i] = take_picture(cam)
        time.sleep(0.1)

    for n in tqdm(range(6), desc="Processing HG fix std modes"):
        mode_group = file.create_group(f"HG_n{n}m{0}")
        images_ds = mode_group.create_dataset("images", (NUM_REP, resolution,height_cam, width_cam), dtype='uint8')

        beam.waist = calculate_fix_std_HG(C, n, z)
        beam.hg(n,0)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)
        for rep in range(NUM_REP):
            stage_z.move_to(z_start-0.001) # in meter
            stage_z.wait_move()
            stage_z.move_to(z_start) # in meter
            stage_z.wait_move()
            for i in range(resolution):
                stage_z.move_to(z_start + i*step_size)
                stage_z.wait_move()
                time.sleep(0.1)
                img = take_picture(cam)
                images_ds[rep, i] = img
            file.flush()